## QC1: DropletUtils

2026-07-21


## Remove empty droplets

Many droplets are empty, mainly consisted of ambient RNA.

We have to filter out these droplets.


1. Construct an ambient pool by summing up UMIs from low-count droplet
2. Test the possibility that ambient RNA accidentally generated this droplet profile
These process are wrapped up by package DropletUtils

In [3]:
library(Seurat)
library(scater)
library(DropletUtils)
library(scran)
library(ggplot2)

set.seed(42) # for reproducibility


data_path <- if (dir.exists("data")) file.path("data", "qc1", "") else file.path("..", "data", "qc1", "")
save_path <- file.path("qc1/")

if (!dir.exists(save_path)) {
  dir.create(save_path)
}


### Load data

In [5]:
raw_sce_list <- list()
sample_numbers <- c(1:9, 12)

for (i in sample_numbers) {
    file_path <- paste0(data_path, sprintf("20094_%04d_A_B/raw_feature_bc_matrix", i))
    raw_sce_list[[paste0("rawsce_", i)]] <- read10xCounts(file_path, type = "sparse", compressed = TRUE)
}

### Examine and check the data

In [9]:
raw_sce_list$rawsce_1

class: SingleCellExperiment 
dim: 36604 6794880 
metadata(1): Samples
assays(1): counts
rownames(36604): ENSG00000243485 ENSG00000237613 ... Htag2 Htag3
rowData names(3): ID Symbol Type
colnames: NULL
colData names(2): Sample Barcode
reducedDimNames(0):
mainExpName: NULL
altExpNames(0):

In [10]:
filtered_sce <- list()

for (i in seq_along(sample_numbers)) {
  rawsce <- raw_sce_list[[i]]
  br.out <- barcodeRanks(counts(rawsce))
  
  e.out <- emptyDrops(counts(rawsce))  # Cells that have UMI counts lower than 100 (by defualt) are empty cells.
  print(table(Sig = e.out$FDR <= 0.05, Limited = e.out$Limited))

  is.cell <- e.out$FDR <= 0.05
  print(sum(is.cell, na.rm = TRUE))

  p <- ggplot(data.frame(br.out), aes(x = rank, y = total)) + 
    geom_point() + 
    scale_x_continuous(trans = "log10") +
    scale_y_continuous(trans = "log10") +
    ggtitle(paste0("raw_sce_", sample_numbers[i])) +
    theme_classic()
  p <- p + 
    geom_hline(aes(yintercept = min(br.out$fitted, na.rm = TRUE),  color = "FDR_0.05"), linetype = "dashed") +
    geom_hline(aes(yintercept = as.numeric(metadata(br.out)$knee), color = "knee"), linetype = "dashed") +
    geom_hline(aes(yintercept = as.numeric(metadata(br.out)$inflection), color = "inflection"), linetype = "dashed") +
    scale_color_manual(
        name = "Cutoffs", 
        values = c(
            "FDR_0.05" = "red",
            "knee" = "dodgerblue",
            "inflection"= "forestgreen"
        )
    ) +
    theme(legend.position = "bottom")
  ggsave(filename = paste0(save_path, "/DropletUtils_", rawsce_list[i], ".png"), plot = p, width = 6, height = 6)
  
  colnames(rawsce) <- colData(rawsce)$Barcode
  rawsce <- rawsce[,which(e.out$FDR <= 0.05)]
  
  filtered_sce[[paste0("DropletUtils_sce_", sample_numbers[i])]] <- rawsce
}

       Limited
Sig     FALSE  TRUE
  FALSE 67355     0
  TRUE   3693  6467
[1] 10160


Warning message in min(br.out$fitted, na.rm = TRUE):
“no non-missing arguments to min; returning Inf”
Warning message in scale_y_continuous(trans = "log10"):
“log-10 transformation introduced infinite values.”


       Limited
Sig     FALSE  TRUE
  FALSE 74890     0
  TRUE    992  6574
[1] 7566


Warning message in min(br.out$fitted, na.rm = TRUE):
“no non-missing arguments to min; returning Inf”
Warning message in scale_y_continuous(trans = "log10"):
“log-10 transformation introduced infinite values.”


       Limited
Sig     FALSE  TRUE
  FALSE 68766     0
  TRUE    469  5881
[1] 6350


...

Warning message in min(br.out$fitted, na.rm = TRUE):
“no non-missing arguments to min; returning Inf”
Warning message in scale_y_continuous(trans = "log10"):
“log-10 transformation introduced infinite values.”


       Limited
Sig     FALSE  TRUE
  FALSE 66869     0
  TRUE    515  2510
[1] 3025


Warning message in min(br.out$fitted, na.rm = TRUE):
“no non-missing arguments to min; returning Inf”
Warning message in scale_y_continuous(trans = "log10"):
“log-10 transformation introduced infinite values.”


       Limited
Sig     FALSE  TRUE
  FALSE 60423     0
  TRUE    948  6294
[1] 7242


Warning message in min(br.out$fitted, na.rm = TRUE):
“no non-missing arguments to min; returning Inf”
Warning message in scale_y_continuous(trans = "log10"):
“log-10 transformation introduced infinite values.”


### Save the result data

In [11]:
for (i in sample_numbers) {
    sce <- filtered_sce[[paste0("DropletUtils_sce_", i)]]
    save(sce, file = paste0(save_path, "/Dropletutils_filtered_sce_", i, ".RData"))
}

Reference
Lun, A. T., Riesenfeld, S., Andrews, T., Gomes, T., & Marioni, J. C. (2019). EmptyDrops: distinguishing cells from empty droplets in droplet-based single-cell RNA sequencing data. Genome biology, 20(1), 1-9.

Pekayvaz, K., Leunig, A., Kaiser, R., Joppich, M., Brambs, S., Janjic, A., ... & Nicolai, L. (2022). Protective immune trajectories in early viral containment of non-pneumonic SARS-CoV-2 infection. Nature communications, 13(1), 1-21.

In [12]:
sessionInfo()

R version 4.5.3 (2026-03-11)
Platform: x86_64-conda-linux-gnu
Running under: Ubuntu 24.04.4 LTS

Matrix products: default
BLAS/LAPACK: /opt/conda/lib/libopenblasp-r0.3.33.so;  LAPACK version 3.12.0

locale:
 [1] LC_CTYPE=C.UTF-8       LC_NUMERIC=C           LC_TIME=C.UTF-8       
 [4] LC_COLLATE=C.UTF-8     LC_MONETARY=C.UTF-8    LC_MESSAGES=C.UTF-8   
 [7] LC_PAPER=C.UTF-8       LC_NAME=C              LC_ADDRESS=C          
[10] LC_TELEPHONE=C         LC_MEASUREMENT=C.UTF-8 LC_IDENTIFICATION=C   

time zone: Etc/UTC
tzcode source: system (glibc)

attached base packages:
[1] stats4    stats     graphics  grDevices utils     datasets  methods  
[8] base     

other attached packages:
 [1] scran_1.38.1                DropletUtils_1.30.0        
 [3] scater_1.38.1               ggplot2_4.0.3              
 [5] scuttle_1.20.0              SingleCellExperiment_1.32.0
 [7] SummarizedExperiment_1.40.0 Biobase_2.70.0             
 [9] GenomicRanges_1.62.1        Seqinfo_1.0.0              
[11